# Data Exploration with Python

**Learning Goal:** Use Claude Code to explore Bolivia's sustainable development data.

This notebook demonstrates:
- Loading data using project configuration
- Basic data inspection
- Summary statistics
- Identifying patterns and relationships

## Setup

First, we load our project configuration and required libraries.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load project configuration
from config import DATA_DIR, OUTPUT_DIR, set_seeds

# Set seeds for reproducibility
set_seeds()

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print(f"Data directory: {DATA_DIR}")

## Load the Datasets

We have 4 datasets that can be joined via `asdf_id`:
1. **regionNames** - Municipality identifiers
2. **sdg** - SDG index scores (0-100)
3. **sdgVariables** - Detailed SDG indicators
4. **ntl** - Night-time lights (economic proxy)

In [ ]:
# Load all datasets
regions = pd.read_csv(DATA_DIR / 'regionNames' / 'regionNames.csv')
sdg = pd.read_csv(DATA_DIR / 'sdg' / 'sdg.csv')
sdg_vars = pd.read_csv(DATA_DIR / 'sdgVariables' / 'sdgVariables.csv')
ntl = pd.read_csv(DATA_DIR / 'ntl' / 'ln_NTLpc.csv')

print(f"Regions: {regions.shape}")
print(f"SDG indices: {sdg.shape}")
print(f"SDG variables: {sdg_vars.shape}")
print(f"Night-time lights: {ntl.shape}")

## Inspect the Data

Let's look at each dataset's structure.

In [ ]:
# Region names - administrative metadata
print("=== REGION NAMES ===")
print(regions.head())
print(f"\nColumns: {list(regions.columns)}")

In [ ]:
# SDG indices - composite scores
print("=== SDG INDICES ===")
print(sdg.head())
print(f"\nColumns: {list(sdg.columns)}")

In [ ]:
# Night-time lights - economic activity proxy
print("=== NIGHT-TIME LIGHTS ===")
print(ntl.head())
print(f"\nColumns: {list(ntl.columns)}")

## Merge Datasets

Create a combined dataset for analysis.

In [ ]:
# Merge all datasets on asdf_id
df = regions.merge(sdg, on='asdf_id')
df = df.merge(ntl, on='asdf_id')

print(f"Combined dataset: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

## Summary Statistics

Explore the distribution of SDG indices across Bolivia's municipalities.

In [ ]:
# SDG index columns
sdg_cols = [col for col in df.columns if col.startswith('index_sdg')]
print(f"SDG indices: {sdg_cols}")

# Summary statistics
df[sdg_cols].describe().round(2)

In [ ]:
# Overall development index (IMDS)
print("=== Municipal Sustainable Development Index (IMDS) ===")
print(df['imds'].describe())
print(f"\nTop 5 municipalities:")
print(df.nlargest(5, 'imds')[['mun', 'dep', 'imds']])

In [ ]:
# Department-level summary
print("=== IMDS by Department ===")
dept_summary = df.groupby('dep')['imds'].agg(['mean', 'std', 'min', 'max', 'count'])
dept_summary = dept_summary.round(2).sort_values('mean', ascending=False)
print(dept_summary)

## Quick Visualizations

In [ ]:
# Distribution of IMDS
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df['imds'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('IMDS Score')
axes[0].set_ylabel('Number of Municipalities')
axes[0].set_title('Distribution of Sustainable Development Index')

# Boxplot by department
df.boxplot(column='imds', by='dep', ax=axes[1], rot=45)
axes[1].set_xlabel('Department')
axes[1].set_ylabel('IMDS Score')
axes[1].set_title('IMDS by Department')
plt.suptitle('')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'imds_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Your Turn!

Try asking Claude Code to help you with:
- "Show me the correlation between SDG indices"
- "Which municipalities have the lowest SDG1 (poverty) scores?"
- "Plot night-time lights over time for a specific department"
- "Compare urban vs rural municipalities"

In [ ]:
# Space for your exploration
